# Text Summarizer — Fine-tuning T5-small on SAMSum

**Pipeline:**
1. Load & inspect the SAMSum dataset
2. Preprocess (clean text)
3. Tokenize for T5
4. Fine-tune with HuggingFace Trainer
5. Save the model
6. Test inference

> **Dataset required:** Place `samsum-train.csv` and `samsum-validation.csv` inside the `ML/data/` folder before running.  
> Download from: https://huggingface.co/datasets/samsum

## 1 — Imports

In [ ]:
import pandas as pd
import torch
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Trainer,
    TrainingArguments,
)
from preprocess import clean_data
from tokenizing import tokenizing_raw_data

print('Torch version :', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('MPS  available:', torch.backends.mps.is_available())

## 2 — Load Data

In [ ]:
train_data = pd.read_csv('data/samsum-train.csv')
valid_data = pd.read_csv('data/samsum-validation.csv')

print('Train shape :', train_data.shape)
print('Valid shape :', valid_data.shape)
train_data.head()

In [ ]:
# Quick sanity-check on first sample
print('--- Dialogue ---')
print(train_data['dialogue'][0])
print()
print('--- Summary ---')
print(train_data['summary'][0])

## 3 — Sample (speeds up training for experimentation)

In [ ]:
# Reduce dataset size for faster training.
# Use the full dataset for best quality.
TRAIN_SAMPLES = 4000
VALID_SAMPLES = 500

train_data = train_data.sample(n=TRAIN_SAMPLES, random_state=42).reset_index(drop=True)
valid_data = valid_data.sample(n=VALID_SAMPLES, random_state=42).reset_index(drop=True)

print('Sampled train:', train_data.shape)
print('Sampled valid:', valid_data.shape)

## 4 — Preprocess

In [ ]:
train_data['dialogue'] = train_data['dialogue'].apply(clean_data)
train_data['summary']  = train_data['summary'].apply(clean_data)

valid_data['dialogue'] = valid_data['dialogue'].apply(clean_data)
valid_data['summary']  = valid_data['summary'].apply(clean_data)

print('Preprocessing done.')
print(train_data['dialogue'][0][:200])

## 5 — Tokenize

In [ ]:
print('Tokenizing training data ...')
train_dataset = train_data.apply(tokenizing_raw_data, axis=1).tolist()

print('Tokenizing validation data ...')
valid_dataset = valid_data.apply(tokenizing_raw_data, axis=1).tolist()

print('Done.')
print('Sample keys:', list(train_dataset[0].keys()))

## 6 — Load Model & Set Device

In [ ]:
# Device selection — MPS (Apple Silicon) > CUDA > CPU
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print('Device:', device)

tokenizer = T5Tokenizer.from_pretrained('t5-small')
model     = T5ForConditionalGeneration.from_pretrained('t5-small')
model.to(device)
print('Model loaded.')

## 7 — Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir                  = './results',
    num_train_epochs            = 6,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size  = 8,
    warmup_steps                = 500,
    weight_decay                = 0.01,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    logging_dir                 = './results/logs',
    logging_steps               = 100,
)

trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = train_dataset,
    eval_dataset  = valid_dataset,
)

print('Trainer ready.')

## 8 — Train

In [ ]:
print('Starting training ...')
trainer.train()
print('Training complete.')

## 9 — Save Model

In [ ]:
SAVE_PATH = './saved_summary_model'

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f'Model saved to {SAVE_PATH}')

## 10 — Test Inference

In [ ]:
def summarize(dialogue: str) -> str:
    """Run inference on a dialogue string."""
    import re

    # Clean
    text = re.sub(r'\r\n|\r|\n', ' ', dialogue)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip().lower()

    # Tokenize
    inputs = tokenizer(
        'summarize: ' + text,
        padding='max_length',
        max_length=512,
        truncation=True,
        return_tensors='pt',
    ).to(device)

    # Generate
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_length=150,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3,
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


# ── Test ──
test_dialogue = """
Reporter: Artificial intelligence continues to expand rapidly across industries.
Recent reports suggest AI adoption has significantly increased over the past few years.

Expert: AI systems are becoming more capable due to advances in deep learning.
These models can now perform complex tasks such as language understanding and code generation.

Reporter: Governments are beginning to introduce regulations to guide AI development.
The goal is to balance innovation with accountability and transparency.

Expert: Ensuring fairness, data privacy, and long-term societal impact are key research areas.
Collaboration between researchers and policymakers will be crucial going forward.
"""

summary = summarize(test_dialogue)
print('Summary:', summary)

---
## Training Results

| Epoch | Train Loss | Val Loss |
|-------|-----------|----------|
| 1     | 3.70      | 0.44     |
| 2     | 0.46      | 0.42     |
| 3     | 0.44      | 0.41     |
| 4     | 0.42      | 0.41     |
| 5     | 0.42      | 0.41     |
| 6     | **0.41**  | **0.41** |
